# Building the fine-tuning dataset

Notebooks 1-4 taught the model *language*. Its corpus was one continuous stream of text —
every message from every chat run together with spaces — so the only thing it could learn is
which token tends to follow which. It has no idea who is speaking, and no idea that a turn
ever *ends*: ask the pre-trained model to generate and it keeps going until `max_new_tokens`
runs out, mid-sentence.

This notebook rebuilds the same chats as discrete, labelled samples, in alternating pairs:

```
<|startoftext|>You<|separator|>hey, are you coming?<|endoftext|>
<|startoftext|>Assistant<|separator|>yeah give me ten minutes<|endoftext|>
```

Same words, three pieces of structure they did not have before:

- `<|startoftext|>` — a sample begins here, and nothing before it is context.
- `<|separator|>` — to its left is *which side is speaking*, to its right is what they said.
  At generation time you write everything up to this marker and let the model continue,
  which is how you ask for a reply instead of a continuation.
- `<|endoftext|>` — the turn is over. This is the id passed as `eos_token_id`, and it is
  what lets generation stop by itself instead of always running to the token limit.

| Stage | What happens |
| --- | --- |
| **Clean** | Notebook 1's ten rules, applied to the same exports |
| **Group** | Consecutive messages from one sender collapse into a single turn |
| **Label** | Every turn becomes `You` or `Assistant`, alternating strictly |
| **Wrap** | Each turn becomes one sample, bounded by the markers above |
| **Verify** | Every sample round-trips, and the whole corpus alternates |
| **Save** | A JSON list of strings for the fine-tuning run to load |

**None of this is new data.** Every message below was already in `combined_text.txt`, so the
model has seen all of it. What is new is the *shape*: which side said it, and where it
stopped.

12,890 messages become 11,719 turns, and those become **8,122 samples — 4,061
`You` → `Assistant` pairs**. That drop is mostly not loss: consecutive turns from the same
side merge into one, and only the two ends of each chat are trimmed away.

## Cleaning, again

Notebook 1 already solved this, and its output is not reusable here. `combined_text.txt` is
one string of messages joined by spaces — the sender was dropped on the way out, because
pre-training does not need it. Fine-tuning is entirely about *who is speaking*, and the
sender is what the role labels are derived from, so the exports have to be read a second time.

Notebooks cannot import each other, so the patterns below are notebook 1's, restated.
(Notebook 4 restates `get_vocab_size` for the same reason.) **If you change a rule, change
it in both places** — otherwise the fine-tuning data stops matching the text the model was
pre-trained on.

One thing is deliberately not carried over: the timestamp. Nothing downstream uses it, and
parsing it was the most error-prone part of notebook 1 — `01/03/2026` is 1 March or
3 January depending on the exporting phone, and guessing wrong corrupts dates with no
exception and no `NaT` to notice. Message order here comes from the file, which WhatsApp
already writes chronologically, so this notebook matches a timestamp only to recognise where
a message starts and then throws it away. No pandas, no format inference, no ambiguity.

In [31]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

# --- Paths ----------------------------------------------------------------
CHAT_DIRECTORY = Path("../Data/private")           # the same exports notebook 1 reads
TOKENIZER_MODEL = Path("../output/tokenizer/my_tokenizer.model")
OUTPUT_PATH = Path("../output/fine_tuning/data/fine_tuning.json")

# --- The sample format ----------------------------------------------------
# The literal strings notebook 2 registered. Their ids are read back off the saved
# tokenizer further down rather than written here, so the two cannot drift apart.
START_OF_TEXT = "<|startoftext|>"
SEPARATOR = "<|separator|>"
END_OF_TEXT = "<|endoftext|>"
MARKERS = (START_OF_TEXT, SEPARATOR, END_OF_TEXT)

# --- The two roles --------------------------------------------------------
# Ordinary text, not special tokens: the model learns them as words like any other.
# What matters is that there are exactly two of them and that they alternate, because
# that is what lets notebook 6 read the corpus two turns at a time.
USER_ROLE = "You"
ASSISTANT_ROLE = "Assistant"

BLOCK_SIZE = 256   # notebook 4's context length; longer samples are truncated when training

# --- Cleaning: notebook 1's rules, compiled once --------------------------
# str.translate() applies the whole table in one C-level pass.
UNICODE_FIXES = str.maketrans({
    "\u202f": " ",   # narrow no-break space (iOS puts it before AM/PM)
    "\u00a0": " ",   # non-breaking space
    "\u200e": None,  # left-to-right mark  -> delete
    "\u200f": None,  # right-to-left mark  -> delete
})

# Rules 1-7: one alternation instead of seven checks, so the line is scanned once.
# (In VERBOSE mode whitespace is ignored, so literal spaces are escaped as "\ ".)
DROP_LINE = re.compile(
    r"""
      Messages\ and\ calls\ are\ end-to-end\ encrypted  # 1 encryption notice
    | <Media\ omitted>                                    # 2 attachments
    | [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}       # 3 email address
    | https?://\S+                                        # 4 link
    | You\ deleted\ this\ message                         # 5 deleted message
    | created\ group                                      # 6 group created
    | added\ you                                          # 7 added to group
    """,
    re.VERBOSE,
)

# Rules 9-10: scrub inside the line, then close the gap the scrub left behind.
INLINE_NOISE = re.compile(r"<This message was edited>|@\w+")
EXTRA_SPACE = re.compile(r"[ \t]{2,}")

# Anchored with ^ and matched via .match(), so a continuation line is rejected on
# its very first character. Handles both export flavours:
#   Android:  14/07/2026, 18:30 - Riya: hey
#   iOS:     [15/07/2026, 9:05 AM] ~ Arjun: hey
MESSAGE_HEADER = re.compile(
    r"""^\[?                                  # iOS wraps the timestamp in [ ]
        (\d{1,2}/\d{1,2}/\d{2,4},\s           # date
         \d{1,2}:\d{2}(?::\d{2})?             # time, seconds optional
         (?:\s?[APap][Mm])?)                   # AM/PM, iOS only
        \]?
        \s?[-~]?\s?                            # " - " on Android, " ~ " on iOS
        ([^:]{1,60}):\                           # sender, up to the first colon
        (.*)$                                   # the message itself
    """,
    re.VERBOSE,
)

### One pass, two lists

The reader walks each file once, keeping senders and bodies in parallel lists. Two details
carry over from notebook 1 unchanged:

**Continuation lines.** A line with no timestamp header is not a new message — it is the
rest of the previous one, so it is appended to the last body collected. A continuation that
arrives before any message has nothing to attach to and is dropped.

**Rule 8 runs last.** `null` is junk only when it is the *whole* message; a line like
`is it null or empty?` has to survive, so it cannot be folded into `DROP_LINE` with the
others. That same final pass drops bodies which scrubbing emptied out — a message that was
nothing but an `@mention` leaves an empty string behind, and an empty turn would teach the
model that `<|separator|>` is sometimes followed immediately by `<|endoftext|>`.

In [32]:
def read_chat(file_path) -> list[tuple[str, str]]:
    """Read one WhatsApp export into (sender, message) pairs, in chat order."""
    text = Path(file_path).read_text(encoding="utf-8").translate(UNICODE_FIXES)

    senders: list[str] = []
    bodies: list[str] = []

    for line in text.splitlines():
        if DROP_LINE.search(line):                     # rules 1-7
            continue

        # Scrub inline noise, then close the gaps it left behind (rules 9-10).
        line = EXTRA_SPACE.sub(" ", INLINE_NOISE.sub("", line)).strip()
        if not line:
            continue

        header = MESSAGE_HEADER.match(line)
        if header:
            _, sender, body = header.groups()          # the timestamp is not needed here
            senders.append(sender.strip())
            bodies.append(body.strip())
        elif bodies:
            # No header: this is the next line of the message we are already building.
            bodies[-1] = f"{bodies[-1]}\n{line}"

    # Rule 8, plus anything scrubbing emptied out.
    return [
        (sender, body)
        for sender, body in zip(senders, bodies)
        if body and body.lower() != "null"
    ]

### Checking the rules actually fire

Two of the ten rules never match anything in `../Data/private` — none of these exports
contains a `created group` or `added you` notice — so a normal run tells you nothing about
whether they still work. That is how notebook 1's `null` rule stayed broken for so long: it
compared `line.split(" ")[-1]` against `readlines()` output, which keeps the trailing `\n`,
so it never matched anything at all, and the 18 `null` bodies in these chats went straight
into the training corpus.

The sample below triggers all ten rules at once, plus a multi-line message and an iOS-format
line. Run it whenever you touch a pattern; it costs milliseconds.

In [33]:
import tempfile

SAMPLE = """\
14/07/2026, 18:30 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
14/07/2026, 18:31 - Riya: <Media omitted>
14/07/2026, 18:32 - Riya: mail me at riya@example.com
14/07/2026, 18:33 - Riya: https://www.example.com/page
14/07/2026, 18:34 - Riya: hey, how are you? <This message was edited>
14/07/2026, 18:35 - Riya: You deleted this message
14/07/2026, 18:36 - Riya: null
14/07/2026, 18:37 - Riya created group "study group"
14/07/2026, 18:38 - Riya added you
14/07/2026, 18:39 - Riya: @arjun are you coming?
14/07/2026, 18:40 - Riya: line one
continued on the next line
[15/07/2026, 9:05 AM] ~ Arjun: iOS format works too
"""

EXPECTED = [
    ("Riya", "hey, how are you?"),
    ("Riya", "are you coming?"),
    ("Riya", "line one\ncontinued on the next line"),
    ("Arjun", "iOS format works too"),
]

sample_file = Path(tempfile.mkdtemp()) / "sample.txt"
sample_file.write_text(SAMPLE, encoding="utf-8")

actual = read_chat(sample_file)

assert actual == EXPECTED, "\n".join(
    ["cleaning rules did not behave as documented:"]
    + [f"  expected {row}" for row in EXPECTED]
    + [f"  actual   {row}" for row in actual]
)
print(f"all 10 rules behave as documented ({len(actual)} messages kept out of 13 lines)")

all 10 rules behave as documented (4 messages kept out of 13 lines)


## One turn per sender, not one per message

People send bursts. Three lines in a row from the same person are one thought, not three,
but the export writes each on its own line:

```
User 1: Hey!
User 1: How are you?
User 2: I am fine
User 2: And you?
User 1: Good.
```

Left alone that becomes five samples, each ending in `<|endoftext|>` after a handful of
words — and the model would learn exactly that: turns are tiny, stop early. Merging each run
into one turn gives it the real distribution instead:

```
User 1: Hey!\nHow are you?
User 2: I am fine\nAnd you?
User 1: Good.
```

The newline earns its place: it keeps the original messages visible as separate lines
without pretending they were separate turns.

**Grouping runs per export, never across the corpus.** Two chats sit next to each other in
file order and have nothing to do with one another, so a turn must never span the boundary
between them — which is why the loop further down calls this once per file.

In [34]:
def group_by_sender(messages: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """Merge each run of consecutive messages from one sender into a single turn."""
    # Parts are collected in a list and joined once at the end. Growing the string with
    # += instead would re-copy the whole turn per message: quadratic in run length.
    runs: list[tuple[str, list[str]]] = []

    for sender, message in messages:
        if runs and runs[-1][0] == sender:
            runs[-1][1].append(message)
        else:
            runs.append((sender, [message]))

    return [(sender, "\n".join(parts)) for sender, parts in runs]

### Checking the grouping

The example from the markdown above, asserted rather than described — including the part
that is easy to get wrong: `User 1` speaks again at the end, and that turn must **not** be
merged back into their earlier one. Only *adjacent* runs collapse.

In [35]:
GROUPING_SAMPLE = [
    ("User 1", "Hey!"),
    ("User 1", "How are you?"),
    ("User 2", "I am fine"),
    ("User 2", "And you?"),
    ("User 1", "Good."),
]

GROUPING_EXPECTED = [
    ("User 1", "Hey!\nHow are you?"),
    ("User 2", "I am fine\nAnd you?"),
    ("User 1", "Good."),
]

grouped = group_by_sender(GROUPING_SAMPLE)

assert grouped == GROUPING_EXPECTED, "\n".join(
    ["grouping did not behave as documented:"]
    + [f"  expected {row}" for row in GROUPING_EXPECTED]
    + [f"  actual   {row}" for row in grouped]
)
assert group_by_sender([]) == [], "an empty chat should produce no turns"

print(f"{len(GROUPING_SAMPLE)} messages -> {len(grouped)} turns, "
      f"and the repeated sender was not merged back")

5 messages -> 3 turns, and the repeated sender was not merged back


## From names to roles

Fine-tuning needs two roles, not fifty-five names. Asking the model to continue
`<|startoftext|>Assistant<|separator|>` only means something if `Assistant` labelled a
consistent side of the conversation during training — so the role has to be *assigned*.

**There is no phone owner in these exports.** No sender appears in more than 5 of the 23
files, and 14 of them are group chats with three to seven participants, so "the person whose
phone this is" does not exist to be found. The rule used here is the most defensible one the
data supports: **in each chat, the most talkative sender becomes the `Assistant`, and
everyone else becomes `You`.**

State the cost plainly, because it is real. Across the 23 chats that makes `Assistant` **20
different people**; 35 of the 55 senders are never the Assistant at all. Worse for anyone
hoping to hear one voice: **10 people sit on both sides of the label** — Meera is the
Assistant in `book_club`, `cousins_group` and `trip_planning`, and is `You` in `exam_prep`
and `startup_founders`. This dataset teaches the *shape* of a reply — one turn, answering the
turn before it, stopping where a reply stops — not one person's voice. If you want the
latter, point `CHAT_DIRECTORY` at a single export.

Two structural details follow from the relabelling, and both matter downstream:

**Adjacent `You` turns have to merge.** In a group chat Arjun and Farah may speak one after
the other; both become `You`, and a `You` turn followed by another `You` turn would pair
`You` with `You` when notebook 6 reads the corpus two turns at a time. `group_by_sender`
already merges adjacent runs sharing a label, so it is reused here rather than rewritten.
This is where most of the 11,719 → 8,122 drop comes from, and no text is lost to it: the
merged bodies are joined, not discarded.

**The ends get trimmed.** A chat opening with `Assistant` has a reply to nothing in front of
it; a chat closing with `You` has a question nobody answered. Dropping both makes every file
start with `You` and end with `Assistant` — which is what keeps the concatenated corpus
alternating *across* file boundaries, not just within them.

In [36]:
def assign_roles(turns: list[tuple[str, str]]) -> tuple[str, list[tuple[str, str]]]:
    """Relabel one chat's turns as You/Assistant, strictly alternating.

    Returns the sender chosen as the Assistant alongside the labelled turns, so the caller
    can report who it picked rather than leaving the reader to guess.
    """
    if not turns:
        return "", []

    counts = Counter(sender for sender, _ in turns)
    # Sorted on (-count, name) rather than Counter.most_common(), which breaks ties by
    # insertion order. The saved dataset has to be identical run to run.
    assistant = sorted(counts, key=lambda name: (-counts[name], name))[0]

    labelled = [
        (ASSISTANT_ROLE if sender == assistant else USER_ROLE, body)
        for sender, body in turns
    ]

    # Two different people speaking in a row both become "You", and a You turn followed by
    # another You turn would pair You with You downstream. group_by_sender already merges
    # adjacent runs sharing a label, so it does this job unchanged - the same function, one
    # level up, grouping on the role instead of the name.
    alternating = group_by_sender(labelled)

    # A leading Assistant turn has no prompt in front of it, and a trailing You turn has no
    # reply after it. Dropping both makes every file open with You and close with Assistant,
    # which is what keeps the *concatenated* corpus alternating across file boundaries too.
    if alternating and alternating[0][0] == ASSISTANT_ROLE:
        alternating = alternating[1:]
    if alternating and alternating[-1][0] == USER_ROLE:
        alternating = alternating[:-1]

    return assistant, alternating

### Checking the role assignment

Four things, on a sample built to break each one: the most talkative sender wins, two
different people speaking in a row collapse into a single `You` turn, the ends are trimmed so
the chat opens with `You` and closes with `Assistant`, and a tie breaks on the name rather
than on dict ordering.

That last one looks pedantic and is not. If a tie resolved by insertion order, the same
export could pick a different Assistant on a different run, and the saved dataset would stop
being reproducible — the same class of failure as notebook 1's unsorted `glob`.

In [37]:
ROLE_SAMPLE = [
    ("Meera", "hey"),                   # Meera speaks twice, so she becomes the Assistant
    ("Arjun", "hi"),                    # Arjun and Farah are both You, and adjacent,
    ("Farah", "hello"),                 # so they have to merge into one turn
    ("Meera", "how are you"),
    ("Naveen", "no reply after this"),  # a trailing You turn: dropped
]

ROLE_EXPECTED = [
    (USER_ROLE, "hi\nhello"),
    (ASSISTANT_ROLE, "how are you"),
]

assistant, labelled = assign_roles(ROLE_SAMPLE)

assert assistant == "Meera", f"expected Meera as the Assistant, got {assistant!r}"
assert labelled == ROLE_EXPECTED, "\n".join(
    ["role assignment did not behave as documented:"]
    + [f"  expected {row}" for row in ROLE_EXPECTED]
    + [f"  actual   {row}" for row in labelled]
)

# The leading Assistant turn ("hey") was dropped, not relabelled.
assert labelled[0][0] == USER_ROLE, "a labelled chat must open with a You turn"
assert labelled[-1][0] == ASSISTANT_ROLE, "a labelled chat must close with an Assistant turn"

# A tie must not depend on dict ordering, or the saved dataset differs run to run.
tied = [("Zoya", "a"), ("Aarav", "b"), ("Zoya", "c"), ("Aarav", "d")]
assert assign_roles(tied)[0] == "Aarav", "a tie must break on the name, not insertion order"

assert assign_roles([]) == ("", []), "an empty chat should produce no turns"

print(f"{len(ROLE_SAMPLE)} turns from 4 senders -> {len(labelled)} alternating turns "
      f"(Assistant = {assistant})")
for role, body in labelled:
    print(f"  {role:<9} {body!r}")

5 turns from 4 senders -> 2 alternating turns (Assistant = Meera)
  You       'hi\nhello'
  Assistant 'how are you'


## Wrapping a turn into a sample

Nothing clever here — the markers are string concatenation:

```
<|startoftext|> + role + <|separator|> + message + <|endoftext|>
```

What makes it work is that the tokenizer maps each marker to **one id**, so the model sees a
single unambiguous symbol rather than a run of ordinary characters it would have to learn to
recognise as a unit.

The role, by contrast, is *not* special — `You` and `Assistant` are ordinary text and encode
as 2 and 4 tokens respectively. That is deliberate: the model already knows those words from
pre-training, and making them special tokens would have meant retraining the tokenizer and
invalidating every notebook upstream. It does mean the role has to survive encode/decode like
any other text, which the check below asserts rather than assumes.

`<|padding|>` and `<|unk|>` were registered in notebook 2 as well but do not appear here:
padding is the data loader's job at fine-tuning time, and nothing is ever out-of-vocabulary
because BPE falls back to raw bytes.

In [38]:
def build_sample(role: str, message: str) -> str:
    """Wrap one turn in the markers the model is being taught to recognise."""
    return f"{START_OF_TEXT}{role}{SEPARATOR}{message}{END_OF_TEXT}"

### The tokenizer has to agree

The format above is only real if the tokenizer treats the markers as special tokens. If it
does not, `<|separator|>` encodes as ordinary characters, the model never gets a clean
boundary signal, and `eos_token_id` matches nothing — generation runs to the token limit
every time, and the failure looks like a training problem rather than a data problem.

So the ids are read off the saved tokenizer rather than hard-coded, and checked.

In [39]:
try:
    import minbpe  # already installed in this environment
except ModuleNotFoundError:
    here = Path.cwd()
    for repo_root in (here, *here.parents):
        if (repo_root / "minbpe" / "minbpe" / "base.py").exists():
            sys.path.insert(0, str(repo_root / "minbpe"))
            print("using minbpe clone at:", repo_root / "minbpe")
            break
    else:
        raise ModuleNotFoundError(
            "minbpe not found. Install it with "
            "`pip install git+https://github.com/karpathy/minbpe.git`"
        )
else:
    print("using installed minbpe:", Path(minbpe.__file__).parent)

from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.load(model_file=str(TOKENIZER_MODEL))

missing = [marker for marker in MARKERS if marker not in tokenizer.special_tokens]
assert not missing, f"{missing} are not special tokens - re-run 2_BytePairEncoding.ipynb"

MARKER_IDS = {marker: tokenizer.special_tokens[marker] for marker in MARKERS}
for marker, idx in MARKER_IDS.items():
    print(f"{idx:>6}  {marker}")

using installed minbpe: /Users/harshpujari/Documents/Work/Aexonic/Projects/llms/minbpe/minbpe
  1024  <|startoftext|>
  1025  <|separator|>
  1026  <|endoftext|>


### Checking one sample end to end

Four assertions on the smallest possible sample: the markers are single ids, they land in
the right places, the text survives a round-trip, and the two halves can be split apart
again — which is what a fine-tuning run does to turn a sample into a prompt and a target.

In [40]:
probe_sample = build_sample(USER_ROLE, "hey, are you coming?")
probe_ids = tokenizer.encode(probe_sample, allowed_special="all")

# 1. the markers are single ids, in the right places
assert probe_ids[0] == MARKER_IDS[START_OF_TEXT], f"first id is {probe_ids[0]}"
assert probe_ids[-1] == MARKER_IDS[END_OF_TEXT], f"last id is {probe_ids[-1]}"
assert probe_ids.count(MARKER_IDS[SEPARATOR]) == 1, "separator is not exactly one id"

# 2. nothing is lost on the way back
assert tokenizer.decode(probe_ids) == probe_sample, "sample did not survive the round-trip"

# 3. the halves can be recovered, which is how a prompt is built at generation time
inner = probe_sample[len(START_OF_TEXT):-len(END_OF_TEXT)]
recovered = tuple(inner.split(SEPARATOR))
assert recovered == (USER_ROLE, "hey, are you coming?"), recovered

# 4. both roles survive the round-trip. They are ordinary text, so nothing guarantees the
#    tokenizer keeps them intact the way it does the markers - it has to be checked.
for role in (USER_ROLE, ASSISTANT_ROLE):
    ids = tokenizer.encode(build_sample(role, "ok"), allowed_special="all")
    decoded_role = tokenizer.decode(ids)[len(START_OF_TEXT):].split(SEPARATOR, 1)[0]
    assert decoded_role == role, f"{role!r} came back as {decoded_role!r}"

print(f"{probe_sample!r}")
print(f"  -> {probe_ids}")
print(f"  {len(probe_ids)} tokens for {len(probe_sample)} characters, "
      f"{len(MARKERS)} of which are markers")
print(f"\nrole label costs: "
      + ", ".join(
          f"{role} = {len(tokenizer.encode(role))} tokens"
          for role in (USER_ROLE, ASSISTANT_ROLE)
      ))

'<|startoftext|>You<|separator|>hey, are you coming?<|endoftext|>'
  -> [1024, 89, 265, 1025, 104, 101, 121, 44, 473, 298, 764, 63, 1026]
  13 tokens for 64 characters, 3 of which are markers

role label costs: You = 2 tokens, Assistant = 4 tokens


## Running it on the real chats

`sorted()` again, for notebook 1's reason: glob order is filesystem-dependent, and an
unsorted read makes the saved dataset differ run to run for no reason.

Both `group_by_sender` and `assign_roles` are called **per file**, never across the corpus.
Grouping per file stops a turn spanning two unrelated conversations; labelling per file means
the Assistant is chosen from the people actually in *that* chat, and that each chat
independently opens with `You` and closes with `Assistant`.

The right-hand column names who was picked. It is worth reading rather than skipping — it is
the one thing here that a change to the input data moves silently, and the two-party chats
(`dad_and_daughter`, `mom_and_son`, `work_besties`) are the ones where the label lines up
with a real person rather than a composite.

In [41]:
chat_files = sorted(CHAT_DIRECTORY.glob("*.txt"))
if not chat_files:
    raise FileNotFoundError(f"no .txt exports found in {CHAT_DIRECTORY.resolve()}")

fine_tuning_data: list[str] = []
total_messages = 0
total_turns = 0

for file in chat_files:
    messages = read_chat(file)
    turns = group_by_sender(messages)                 # per file, so a turn never spans two chats
    assistant, labelled = assign_roles(turns)         # also per file, for the same reason

    fine_tuning_data.extend(build_sample(role, body) for role, body in labelled)
    total_messages += len(messages)
    total_turns += len(turns)

    print(f"{file.stem:<22} {len(messages):>5} messages -> {len(turns):>5} turns "
          f"-> {len(labelled):>5} labelled   Assistant = {assistant}")

print(f"\n{total_messages:,} messages from {len(chat_files)} chats "
      f"-> {total_turns:,} turns -> {len(fine_tuning_data):,} samples "
      f"({len(fine_tuning_data) // 2:,} You -> Assistant pairs)")

book_club                386 messages ->   372 turns ->   202 labelled   Assistant = Meera
bro_chat                 927 messages ->   906 turns ->   496 labelled   Assistant = Vikram
college_project          904 messages ->   861 turns ->   504 labelled   Assistant = Neha
cousins_group            495 messages ->   482 turns ->   218 labelled   Assistant = Meera
cricket_gang             471 messages ->   469 turns ->   218 labelled   Assistant = Karan
dad_and_daughter         576 messages ->   471 turns ->   470 labelled   Assistant = Papa
exam_prep                498 messages ->   464 turns ->   266 labelled   Assistant = Priya
family_group             917 messages ->   886 turns ->   366 labelled   Assistant = Priya
foodie_chat              500 messages ->   407 turns ->   406 labelled   Assistant = Naina
freelance_client         381 messages ->   304 turns ->   302 labelled   Assistant = Kabir Studio
gym_buddies              367 messages ->   274 turns ->   274 labelled   Assistant =

### Checking every sample, not a few

Three things can go wrong across the whole set, and all of them are quiet.

**A forged marker.** If a message body literally contained `<|endoftext|>`, its sample would
carry two of them and fine-tuning would read the tail as a second turn with no role. This is
why minbpe defaults to `allowed_special="none_raise"` — encoding user text as control tokens
lets anyone plant a boundary. Counting markers per sample catches it. If it ever fires, the
fix is a decision (scrub the text? drop the message?) rather than something to paper over
here.

**A sample that does not survive encoding.** Checked on all of them rather than a spot
sample, because it costs a few seconds and the failure — one bad turn among eight thousand —
is invisible any other way.

**A break in the alternation.** This is the one notebook 6 depends on outright: it reads the
saved list two samples at a time, so if a single `You` turn were ever followed by another
`You`, every pair after it would be off by one — a prompt matched to the wrong reply, all the
way to the end of the corpus. Nothing would crash. The loss would fall, more slowly than it
should, and the model would learn to answer questions it was never shown. Asserting the whole
sequence costs milliseconds.

In [42]:
for sample in fine_tuning_data:
    counts = tuple(sample.count(marker) for marker in MARKERS)
    assert counts == (1, 1, 1), (
        f"expected one of each marker, got {dict(zip(MARKERS, counts))} "
        f"in {sample[:80]!r}"
    )

token_lengths = []
roles = []
for sample in fine_tuning_data:
    ids = tokenizer.encode(sample, allowed_special="all")
    assert tokenizer.decode(ids) == sample, f"round-trip failed on {sample[:80]!r}"
    token_lengths.append(len(ids))
    roles.append(sample[len(START_OF_TEXT):].split(SEPARATOR, 1)[0])

# The invariant notebook 6 is built on: the whole corpus alternates You, Assistant, You,
# ... so reading it two samples at a time never pairs a prompt with the wrong reply, and
# never straddles the boundary between two chats.
assert len(roles) % 2 == 0, f"odd sample count ({len(roles):,}) - they cannot all pair up"
expected_roles = [USER_ROLE, ASSISTANT_ROLE] * (len(roles) // 2)
first_break = next(
    (i for i, (actual, wanted) in enumerate(zip(roles, expected_roles)) if actual != wanted),
    None,
)
assert first_break is None, (
    f"alternation breaks at sample {first_break}: expected "
    f"{expected_roles[first_break]!r}, got {roles[first_break]!r}"
)

print(f"all {len(fine_tuning_data):,} samples carry exactly one of each marker "
      "and survive encode/decode,")
print(f"and alternate {USER_ROLE} -> {ASSISTANT_ROLE} cleanly across all "
      f"{len(fine_tuning_data) // 2:,} pairs")

all 8,122 samples carry exactly one of each marker and survive encode/decode,
and alternate You -> Assistant cleanly across all 4,061 pairs


### How much fine-tuning signal is there?

Worth reading honestly. The **258,369** tokens here are more than the **195,469** notebook 2
counted for the same messages, and none of the difference is new content: every turn now pays
for three markers and a role label, and each turn is encoded on its own, so BPE merges no
longer run across message boundaries the way they did in one continuous stream.

The role labels happen to cost exactly what the markers cost — `You` is 2 tokens and
`Assistant` is 4, averaging 3 per sample against the markers' 3. That is arithmetic
coincidence rather than a design property; rename either role and the two columns come apart.

The number to watch is the longest **pair**, not the longest sample. Notebook 6 concatenates
a `You` turn with its `Assistant` reply into a single training sequence, so `block_size` has
to hold both. Truncation there keeps the *end* of the pair, which protects the `<|endoftext|>`
this whole exercise exists to teach — the question is what gets clipped instead.

In [43]:
total_tokens = sum(token_lengths)
marker_tokens = len(MARKERS) * len(token_lengths)
role_tokens = sum(len(tokenizer.encode(role)) for role in roles)

print(f"{len(fine_tuning_data):>9,} samples  ({len(fine_tuning_data) // 2:,} pairs)")
print(f"{total_tokens:>9,} tokens")
print(f"{total_tokens / len(token_lengths):>9.1f} tokens per sample on average "
      f"(shortest {min(token_lengths)}, longest {max(token_lengths)})\n")

for label, count in (
    ("markers", marker_tokens),
    ("role labels", role_tokens),
    ("message text", total_tokens - marker_tokens - role_tokens),
):
    print(f"  {label:<13}{count:>9,}  {count / total_tokens:>6.1%}")

# What matters downstream is the *pair*, since notebook 6 concatenates prompt and reply
# into one training sequence. A pair over block_size loses its ending when truncated.
pair_lengths = [
    token_lengths[i] + token_lengths[i + 1] for i in range(0, len(token_lengths), 2)
]
too_long = sum(1 for n in pair_lengths if n > BLOCK_SIZE)
print(f"\n{max(pair_lengths)} tokens in the longest You+Assistant pair")
print(f"{too_long} of {len(pair_lengths):,} pairs exceed block_size={BLOCK_SIZE} "
      "and would lose their ending")

    8,122 samples  (4,061 pairs)
  258,369 tokens
     31.8 tokens per sample on average (shortest 6, longest 335)

  markers         24,366    9.4%
  role labels     24,366    9.4%
  message text   209,637   81.1%

377 tokens in the longest You+Assistant pair
2 of 4,061 pairs exceed block_size=256 and would lose their ending


## Saving the dataset

A JSON list of strings — the simplest thing that survives a round-trip without a schema.
`ensure_ascii=False` keeps emoji and non-Latin text readable in the file instead of turning
them into `\uXXXX` escapes; both forms load back to the same string, but only one of them
can be read in an editor.

In [44]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(
    json.dumps(fine_tuning_data, ensure_ascii=False, indent=4), encoding="utf-8"
)

print(f"wrote {OUTPUT_PATH.resolve()}")
print(f"  {len(fine_tuning_data):,} samples, {OUTPUT_PATH.stat().st_size:,} bytes")

wrote /Users/harshpujari/Documents/Work/Aexonic/Projects/llms/TrainYourOwnLLM-Tutorial/output/fine_tuning/data/fine_tuning.json
  8,122 samples, 1,017,710 bytes


And read it back exactly as the fine-tuning notebook will. A dataset that is correct in
memory but not on disk breaks a notebook later, where the cause is invisible.

In [45]:
reloaded = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))

assert reloaded == fine_tuning_data, "the file does not match what was built in memory"
assert all(isinstance(sample, str) for sample in reloaded), "every sample must be a string"

print(f"reloaded {len(reloaded):,} samples, identical to the ones in memory\n")
print("first sample:")
print(f"  {reloaded[0]}")
print("\nlongest sample, truncated for display:")
print(f"  {reloaded[token_lengths.index(max(token_lengths))][:110]}...")

reloaded 8,122 samples, identical to the ones in memory

first sample:
  <|startoftext|>You<|separator|>yes please, i've been in a reading slump since november
same honestly, i finished maybe half a book last month
proposal: something with a bit of weight to it, not too long though, we always overestimate how fast we read<|endoftext|>

longest sample, truncated for display:
  <|startoftext|>You<|separator|>yes due end of March, vendor quote is 18,000 per lift annually
two vendors quot...


### Where to go next

- **This dataset teaches format, not facts.** The model saw every one of these messages
  during pre-training. Fine-tuning on them installs the turn structure — expect the loss to
  start below the pre-training loss and fall quickly, and do not read that as the model
  getting smarter.
- **`Assistant` is 20 people, not one.** The label marks *a side of the conversation*, so the
  model learns to produce a plausible reply, not to imitate anyone. A dataset built from a
  single two-party export is far smaller and far more coherent, because the label then means
  one person.
- **Re-run this notebook if you change notebook 1's rules.** The two cleaning
  implementations have to stay identical, or fine-tuning drifts away from pre-training.
- **It overfits more slowly than the size suggests.** The obvious worry is that 4,061 pairs of
  already-seen text will be memorised within an epoch. Notebook 6 measures it: after three
  epochs validation was still falling at every evaluation, train 2.66 against validation 2.92,
  and early stopping never fired. Watch the curve rather than assuming — but do not stop short
  on the assumption that it must be overfitting by now.
- **Changing the role rule invalidates the saved file.** `assign_roles` decides the shape of
  every downstream batch; if you edit it, re-run this notebook before notebook 6, or notebook
  6 will silently train on the previous dataset.